# Projet HAR — Version 1 : réseau à une seule couche cachée

**Master 2 MALIA — Deep Learning — Université Lumière Lyon 2**
Binôme : DIOP Alioune, FALL Madjiguène — responsable de ce notebook : DIOP Alioune

## Objectifs
Ce notebook traite les parties 1.3, 1.4 et 2 du sujet, puis l'évaluation finale de la version 1 :
1. construire un ensemble de **validation par volontaires entiers** et montrer pourquoi un découpage aléatoire des
   lignes est trompeur (partie 1.3) ;
2. entraîner une **référence** simple, la régression logistique multiclasse (partie 1.4) ;
3. construire un MLP à **une couche cachée** de 64 neurones, le **vérifier avant d'entraîner** (perte initiale,
   surapprentissage d'un mini-batch), puis comparer **activations** et **optimiseurs**, chacun avec un **pas
   d'apprentissage réglé** (partie 2.2) ;
4. mesurer la **variabilité** due à la graine aléatoire et n'en tirer que des conclusions justifiées (partie 2.3) ;
5. évaluer **une seule fois** sur le test la référence et la configuration retenue.

On suit la méthodologie du cours (*A recipe for training neural networks*) : objectif et référence → regarder les
données (notebook `00_exploration`) → modèle simple vérifié de bout en bout → surapprendre un mini-batch → régler →
**tester une seule fois**. Les fonctions `train_epoch`, `evaluate`, `fit`, `overfit_one_batch` et `lr_range_test`
reprennent celles des TP 1 et 2.

## Plan (structure imposée par le sujet, section 8.1)
| Section | Contenu | Partie du sujet |
|---|---|---|
| 1 | Imports et mise en place | |
| 2 | Chargement des données | 1.1–1.2 |
| 3 | Découpage apprentissage / validation, référence, DataLoaders | 1.3, 1.4 |
| 4 | Modèle, fonctions d'entraînement et d'évaluation, contrôles | 2.1 |
| 5 | Boucle d'entraînement principale : pas, activations, optimiseurs, graines | 2.2, 2.3 |
| 6 | Visualisation et comparaison des résultats | 2.2 |
| 7 | Évaluation finale sur le jeu de test | 1.3, 5.2 (V1) |

**Temps de calcul.** Sur un simple processeur (CPU), le notebook complet s'exécute en 15 à 25 minutes ; les cellules
les plus longues sont le balayage du pas (5.2) et les entraînements répétés (5.3).

---
## Partie 1 — Imports et mise en place

Comme dans les TP, la cellule suivante choisit le meilleur matériel disponible et définit `set_seed`, qui fixe les
graines aléatoires : elle permet de comparer deux configurations **à initialisation égale** et rend les résultats
reproductibles.

In [ ]:
# Installation éventuelle (Google Colab ou machine personnelle) :
# !pip install torch scikit-learn pandas matplotlib seaborn

import os, json, copy, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

def set_seed(seed=0):
    torch.manual_seed(seed); np.random.seed(seed)

set_seed(0)
if torch.cuda.is_available():                    # GPU NVIDIA
    device = torch.device("cuda")
elif torch.backends.mps.is_available():          # GPU des Mac à puce Apple
    device = torch.device("mps")
else:                                            # processeur
    device = torch.device("cpu")
print("PyTorch", torch.__version__, "| device :", device)

DATA_DIR = os.environ.get("HAR_DATA_DIR", "../data/HAR")
RESULTS_DIR = os.environ.get("HAR_RESULTS_DIR", "../results")
FIG_DIR, LOG_DIR = os.path.join(RESULTS_DIR, "figures"), os.path.join(RESULTS_DIR, "logs")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(LOG_DIR, exist_ok=True)

def save_fig(name):
    # Enregistre la figure courante (pour le rapport), puis l'affiche.
    plt.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight")
    plt.show()

CLASSES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]
SEEDS = [0, 1, 2]                 # graines des entraînements répétés (partie 2.3 : au moins 3)
N_EPOCHS = 50                     # durée des entraînements principaux
N_EPOCHS_SWEEP = 15               # durée des entraînements du balayage du pas
if os.environ.get("HAR_FAST") == "1":            # uniquement pour tester rapidement que le notebook s'exécute
    N_EPOCHS, N_EPOCHS_SWEEP = 6, 3

---
## Partie 2 — Chargement des données

L'exploration complète se trouve dans `00_exploration.ipynb`. On retient ici ce qui sert à la modélisation :
561 variables d'entrée numériques, sans valeur manquante, déjà comprises dans $[-1, 1]$ ; une cible à 6 classes,
proches de l'équilibre ; la colonne `subject`, qui **ne sert pas d'entrée** mais servira à construire la validation.

La cible est encodée par un **indice de classe entier** (0 à 5), le format attendu par `nn.CrossEntropyLoss`
(TP 1, section 4.2).

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
features = [c for c in df.columns if c not in ("subject", "Activity")]
label_to_int = {c: i for i, c in enumerate(CLASSES)}

X_all = df[features].to_numpy(dtype=np.float32)
y_all = df["Activity"].map(label_to_int).to_numpy(dtype=np.int64)
s_all = df["subject"].to_numpy()
print("X :", X_all.shape, X_all.dtype, "| y :", y_all.shape, y_all.dtype, "| volontaires :", len(np.unique(s_all)))
print("effectif par classe :", dict(zip(CLASSES, np.bincount(y_all))))

---
## Partie 3 — Découpage apprentissage / validation, référence et DataLoaders

Cours (*Training, validation and test sets*) : l'**apprentissage** sert à calculer les gradients, la **validation**
à faire **tous** les choix (architecture, activation, optimiseur, pas, nombre d'époques), le **test** une seule fois,
à la fin. `test.csv` ne sera donc ouvert qu'en partie 7.

### 3.1 Pourquoi réserver des volontaires entiers ?
Chaque fenêtre dure 2,56 s et deux fenêtres consécutives se **chevauchent de 50 %** : elles partagent la moitié de
leurs mesures, et proviennent de la même personne, dans la même activité. Si l'on tire les lignes au hasard, presque
chaque fenêtre de validation a une « jumelle » dans l'apprentissage : le modèle est évalué sur des exemples qu'il a
presque vus. C'est une **fuite d'information** (*data leakage*). Or le test contient des **personnes nouvelles**.

On construit donc deux découpages de même taille, que l'on comparera :
- **par volontaire** : 5 volontaires entiers de `train.csv` forment la validation (c'est le découpage retenu) ;
- **aléatoire** : le même nombre de lignes, tirées au hasard (stratifié par classe).

In [ ]:
rng = np.random.default_rng(0)
val_subjects = np.sort(rng.choice(np.unique(s_all), size=5, replace=False))
va_idx = np.where(np.isin(s_all, val_subjects))[0]             # découpage par volontaire
tr_idx = np.where(~np.isin(s_all, val_subjects))[0]
assert len(set(s_all[tr_idx]) & set(s_all[va_idx])) == 0      # aucun volontaire commun

tr_idx_r, va_idx_r = train_test_split(np.arange(len(y_all)), test_size=len(va_idx),   # découpage aléatoire
                                      random_state=0, stratify=y_all)

print("volontaires de validation :", val_subjects)
print(f"par volontaire : {len(tr_idx)} fenêtres d'apprentissage, {len(va_idx)} de validation ({len(va_idx)/len(y_all):.1%})")
print(f"aléatoire      : {len(tr_idx_r)} fenêtres d'apprentissage, {len(va_idx_r)} de validation")
pd.DataFrame({"apprentissage": np.bincount(y_all[tr_idx]), "validation": np.bincount(y_all[va_idx])}, index=CLASSES)

### 3.2 Normalisation des entrées
Les variables sont déjà dans $[-1, 1]$, mais leurs moyennes et écarts-types restent différents (exploration,
partie 4). On les **standardise**, $x_j \leftarrow (x_j-\mu_j)/\sigma_j$ (cours : *Data preparation*), avec des
statistiques calculées **sur l'apprentissage uniquement**, puis appliquées à la validation et, plus tard, au test
(TP 2, section 1.8).

**Exercice 3.2.1** — Pourquoi recalcule-t-on des statistiques différentes pour le découpage aléatoire ?

In [ ]:
def standardize(X_train, *others):
    # Statistiques de l'APPRENTISSAGE uniquement, appliquées à tous les ensembles.
    mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
    sd[sd == 0] = 1.0                                          # variable constante : on ne divise pas par 0
    return [((X - mu) / sd).astype(np.float32) for X in (X_train,) + others], (mu, sd)

(Xtr, Xva), (mu_x, sd_x) = standardize(X_all[tr_idx], X_all[va_idx])           # par volontaire
ytr, yva = y_all[tr_idx], y_all[va_idx]
(Xtr_r, Xva_r), (mu_r, sd_r) = standardize(X_all[tr_idx_r], X_all[va_idx_r])  # aléatoire
ytr_r, yva_r = y_all[tr_idx_r], y_all[va_idx_r]

print("apprentissage : moyennes", Xtr.mean(axis=0)[:4].round(2), "| écarts-types", Xtr.std(axis=0)[:4].round(2))
print("validation (proches de 0 et 1, sans l'être exactement) :", Xva.mean(axis=0)[:4].round(2), Xva.std(axis=0)[:4].round(2))

*Réponse :*

### 3.3 Les « fenêtres jumelles » : une vérification directe
Pour chaque fenêtre de validation, on cherche la fenêtre d'apprentissage **la plus proche** (distance euclidienne,
entrées standardisées) et l'on regarde si elle provient du même volontaire.

In [ ]:
jumelles = {}
for name, (Xa, Xb, ia, ib) in {"aléatoire": (Xtr_r, Xva_r, tr_idx_r, va_idx_r),
                               "par volontaire": (Xtr, Xva, tr_idx, va_idx)}.items():
    dist, nn_idx = NearestNeighbors(n_neighbors=1).fit(Xa).kneighbors(Xb)
    jumelles[name] = {
        "distance médiane au plus proche voisin": round(float(np.median(dist)), 2),
        "plus proche voisin = même volontaire": f"{(s_all[ia][nn_idx[:, 0]] == s_all[ib]).mean():.1%}",
        "plus proche voisin = même activité": f"{(y_all[ia][nn_idx[:, 0]] == y_all[ib]).mean():.1%}",
    }
pd.DataFrame(jumelles)

**Question 3.3.1** — Comparez les deux découpages. Que montre la proportion de plus proches voisins issus du même
volontaire ? Quelle estimation de la performance sera la plus optimiste, et pourquoi ?

*Réponse :*

### 3.4 La référence : régression logistique multiclasse (partie 1.4)
Cours (*A recipe for training neural networks*, étape 1, et TP 2, section 1.3) : toujours commencer par une
**référence simple**. La régression logistique multinomiale est un réseau **sans couche cachée** : une couche
linéaire $561 \to 6$ suivie de la softmax, entraînée par entropie croisée. Chaque réseau sera comparé à elle.

On mesure l'**accuracy** et le **F1-score macro** (moyenne des F1 des 6 classes), ainsi que le F1 de chaque classe.

In [ ]:
t0 = time.time()
baseline = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
yva_pred_base = baseline.predict(Xva)
base_val = {"accuracy": accuracy_score(yva, yva_pred_base), "f1_macro": f1_score(yva, yva_pred_base, average="macro")}
print(f"référence (validation) : accuracy {base_val['accuracy']:.4f} | F1 macro {base_val['f1_macro']:.4f} "
      f"| {baseline.coef_.size + baseline.intercept_.size} paramètres | {time.time() - t0:.1f} s")

display(pd.DataFrame({"F1 (validation)": f1_score(yva, yva_pred_base, average=None)}, index=CLASSES).round(3))
ConfusionMatrixDisplay(confusion_matrix(yva, yva_pred_base, normalize="true"), display_labels=CLASSES).plot(
    cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=45)
plt.title("Référence — matrice de confusion (validation)"); save_fig("v1_reference_confusion_val")

### 3.5 Découpage aléatoire ou par volontaire : même modèle, deux estimations (partie 1.3)
On entraîne **le même modèle** (la régression logistique) avec chacun des deux découpages et l'on compare les
scores de validation. La comparaison avec le **test** sera faite en partie 7, une fois tous les choix figés.
Elle sera refaite avec le MLP retenu en section 5.5.

In [ ]:
baseline_r = LogisticRegression(max_iter=2000).fit(Xtr_r, ytr_r)
yva_r_pred = baseline_r.predict(Xva_r)
split_val = {("régression logistique", "aléatoire"): accuracy_score(yva_r, yva_r_pred),
             ("régression logistique", "par volontaire"): base_val["accuracy"]}
pd.Series(split_val, name="accuracy de validation").round(4).to_frame()

**Question 3.5.1** — Quel découpage donne le meilleur score de validation ? Est-ce le signe d'un meilleur modèle ?

*Réponse :*

### 3.6 Les DataLoaders
Comme au TP 1 (partie 4) : `TensorDataset` associe entrées et étiquettes, `DataLoader` les renvoie par mini-batches,
**mélangés** à chaque époque pour l'apprentissage seulement. Les entrées sont en `float32`, les étiquettes en
`torch.long` (indices de classe pour `CrossEntropyLoss`). Le générateur à graine fixe rend l'ordre des mini-batches
reproductible.

In [ ]:
def make_loaders(Xtr, ytr, Xva, yva, batch_size=64, seed=0):
    tr_ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.long))
    va_ds = TensorDataset(torch.tensor(Xva, dtype=torch.float32), torch.tensor(yva, dtype=torch.long))
    g = torch.Generator().manual_seed(seed)
    return (DataLoader(tr_ds, batch_size=batch_size, shuffle=True, generator=g),
            DataLoader(va_ds, batch_size=1024, shuffle=False))

train_loader, val_loader = make_loaders(Xtr, ytr, Xva, yva)
Xb, yb = next(iter(train_loader))
print("un mini-batch :", Xb.shape, Xb.dtype, "|", yb.shape, yb.dtype)
print("mises à jour des poids par époque :", len(train_loader))

---
## Partie 4 — Modèle, fonctions d'entraînement et d'évaluation, contrôles (partie 2.1)

### 4.1 Le modèle `MLP1`
Même structure que `SimpleMLP` au TP 1 : une couche linéaire entrée → couche cachée, une activation non linéaire,
une couche linéaire couche cachée → sortie. La méthode `forward` renvoie les **logits** : pas de softmax dans le
modèle, `nn.CrossEntropyLoss` l'applique elle-même (TP 1, section 6.5 : le bogue de la double softmax).

La méthode `hidden` renvoie les activations de la couche cachée ; elle servira à l'interprétation (partie 6 du sujet).

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}

class MLP1(nn.Module):
    def __init__(self, input_dim=561, hidden_dim=64, output_dim=6, activation="relu"):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = ACTIVATIONS[activation]()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def hidden(self, x):
        return self.act(self.fc1(x))          # représentation apprise par la couche cachée

    def forward(self, x):
        return self.fc2(self.hidden(x))       # logits

def count_params(model):
    return sum(p.numel() for p in model.parameters())

set_seed(0)
model = MLP1()
print(model)
print("nombre de paramètres :", count_params(model))
for name, p in model.named_parameters():
    print(f"  {name:12s} {tuple(p.shape)}")

**Question 4.1.1** — Retrouvez le nombre de paramètres à la main. Comparez-le à celui de la régression logistique
(section 3.4).

*Réponse :*

### 4.2 Les fonctions communes
`train_epoch` est celle du TP 1 (les cinq étapes, la somme des erreurs divisée par `len(loader.dataset)`).
`evaluate` est celle du TP 1, complétée par le **F1-score macro** : on garde les prédictions de tous les
mini-batches pour le calculer avec `sklearn` sur l'ensemble complet (le F1 ne se moyenne pas mini-batch par
mini-batch). `fit` reprend `fit_clf` du TP 2 : à chaque époque, perte, accuracy et F1 en apprentissage et en
validation.

In [ ]:
def train_epoch(model, loader, optimizer, lossfn, device):
    model.train()
    total_loss = 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = lossfn(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
    return total_loss / len(loader.dataset)

def predict(model, loader, device):
    # Renvoie (vraies classes, classes prédites) sur tout le loader, en tableaux NumPy.
    model.eval()
    ys, preds = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(device)).argmax(dim=1).cpu())
            ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(preds).numpy()

def evaluate(model, loader, lossfn, device):
    # TP 1 + F1 : renvoie (erreur moyenne, accuracy, F1 macro)
    model.eval()
    total_loss, ys, preds = 0.0, [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            total_loss += lossfn(logits, y).item() * X.size(0)
            preds.append(logits.argmax(dim=1).cpu()); ys.append(y.cpu())
    y_true, y_pred = torch.cat(ys).numpy(), torch.cat(preds).numpy()
    return total_loss / len(loader.dataset), accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")

def fit(model, train_loader, val_loader, optimizer, n_epochs, lossfn=None, verbose=True):
    # Entraîne un classifieur et garde l'historique, époque par époque.
    lossfn = nn.CrossEntropyLoss() if lossfn is None else lossfn
    hist = {"train_loss": [], "train_acc": [], "train_f1": [], "val_loss": [], "val_acc": [], "val_f1": []}
    t0 = time.time()
    for ep in range(1, n_epochs + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, lossfn, device)
        _, tr_acc, tr_f1 = evaluate(model, train_loader, lossfn, device)
        va_loss, va_acc, va_f1 = evaluate(model, val_loader, lossfn, device)
        for k, v in zip(hist, [tr_loss, tr_acc, tr_f1, va_loss, va_acc, va_f1]):
            hist[k].append(v)
        if verbose and (ep == 1 or ep % 10 == 0):
            print(f"époque {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} F1 {va_f1:.4f}")
        if not math.isfinite(tr_loss):
            print("  -> l'erreur n'est plus un nombre fini : arrêt."); break
    hist["best_epoch"] = int(np.argmin(hist["val_loss"])) + 1      # époque où la perte de validation est minimale
    hist["time"] = time.time() - t0
    return hist

def plot_curves(hists, key_pairs=(("train_loss", "val_loss"), ("train_acc", "val_acc")), title="", name=None):
    # TP 2 : tirets = apprentissage, traits pleins = validation
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    for ax, (ktr, kva) in zip(axes, key_pairs):
        for i, (lab, h) in enumerate(hists.items()):
            ax.plot(range(1, len(h[ktr]) + 1), h[ktr], ls="--", c=f"C{i}", alpha=0.7)
            ax.plot(range(1, len(h[kva]) + 1), h[kva], ls="-", c=f"C{i}", label=lab)
        ax.set_xlabel("époque"); ax.set_ylabel(kva.replace("val_", "")); ax.legend(fontsize=8)
    axes[0].set_title(title + "  (tirets : apprentissage, traits pleins : validation)", fontsize=9)
    plt.tight_layout()
    save_fig(name) if name else plt.show()

### 4.3 Premier contrôle : la perte avant tout apprentissage
Cours (*Sanity checks before long trainings*) et TP 2 (section 2.2) : à l'initialisation, les logits sont petits,
la softmax est presque uniforme sur les $K=6$ classes, et la perte doit être proche de $-\ln(1/6)=\ln 6$.

In [ ]:
lossfn = nn.CrossEntropyLoss()
print(f"valeur attendue : ln(6) = {math.log(6):.3f}")
for seed in SEEDS:
    set_seed(seed)
    m = MLP1().to(device)
    m.eval()
    with torch.no_grad():
        logits = m(torch.tensor(Xva, dtype=torch.float32).to(device))
    loss0, acc0, _ = evaluate(m, val_loader, lossfn, device)
    print(f"graine {seed} : perte initiale {loss0:.3f} | écart-type des logits {logits.std().item():.3f} | accuracy {acc0:.3f}")

**Question 4.3.1** — La perte initiale est-elle proche de la valeur attendue ? Quelle accuracy attend-on avant
entraînement ? Que signifierait une perte initiale beaucoup plus grande (TP 2, question 2.2.1) ?

*Réponse :*

### 4.4 Deuxième contrôle : surapprendre un seul mini-batch
Cours (*Sanity checks before long trainings*) et TP 2 (section 2.3) : un réseau correct doit pouvoir **apprendre
par cœur** un tout petit nombre d'exemples. On l'entraîne sur **un seul** mini-batch de 64 fenêtres, toujours le
même, pendant 300 itérations, sans régularisation ; la perte doit tendre vers 0.

In [ ]:
def overfit_one_batch(model, X, y, n_iter=300, lr=1e-3):
    lossfn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses = []
    for _ in range(n_iter):
        optimizer.zero_grad()
        loss = lossfn(model(X), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

X64, y64 = next(iter(make_loaders(Xtr, ytr, Xva, yva, batch_size=64, seed=0)[0]))
X64, y64 = X64.to(device), y64.to(device)
set_seed(0)
m = MLP1().to(device)
l_batch = overfit_one_batch(m, X64, y64)
with torch.no_grad():
    acc_batch = (m(X64).argmax(dim=1) == y64).float().mean().item()
print(f"perte : {l_batch[0]:.3f} -> {l_batch[-1]:.5f} | accuracy finale sur le mini-batch : {acc_batch:.0%}")
plt.figure(figsize=(7, 3.2))
plt.semilogy(l_batch)
plt.xlabel("itération"); plt.ylabel("perte sur le mini-batch (log)"); plt.title("Surapprentissage d'un mini-batch de 64 fenêtres")
save_fig("v1_surapprentissage_minibatch")

**Question 4.4.1** — Le test est-il réussi ? Que faudrait-il conclure s'il échouait (TP 2, question 2.3.2) ? Ce test
dit-il quelque chose de la capacité de généralisation du réseau ?

*Réponse :*

---
## Partie 5 — Boucle d'entraînement principale (parties 2.2 et 2.3)

On compare **3 activations** (ReLU, Sigmoid, Tanh) et **3 optimiseurs** (SGD, SGD avec momentum 0.9, Adam).
Le sujet demande que le pas d'apprentissage soit **réglé pour chaque optimiseur** : comparer SGD et Adam avec le
même pas n'aurait pas de sens (cours : *The main hyperparameters*, le pas est l'hyperparamètre de priorité ⋆⋆⋆).
On procède comme au TP 2 (section 1.10) : un *LR range test* pour situer la bonne zone, puis un **balayage** sur une
grille logarithmique pour trancher.

In [ ]:
OPTIMIZERS = {
    "SGD":          lambda p, lr: torch.optim.SGD(p, lr=lr),
    "SGD+momentum": lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9),
    "Adam":         lambda p, lr: torch.optim.Adam(p, lr=lr),
}

### 5.1 Le *LR range test*
TP 2 (section 1.10) : une seule époque, le pas part de $\eta_{\min}=10^{-6}$ et est multiplié par un facteur constant
$q$ à chaque mini-batch pour atteindre $\eta_{\max}=10$ ; on trace la perte lissée et l'on s'arrête dès qu'elle
explose. Avec $n$ mini-batches, $\eta_{\min}\,q^{\,n-1}=\eta_{\max}$, donc $q=(\eta_{\max}/\eta_{\min})^{1/(n-1)}$.
Une époque ne compte ici qu'une centaine de mini-batches : on utilise des mini-batches de 16 pour avoir une courbe
plus fine.

In [ ]:
def lr_range_test(model, loader, lossfn, make_optimizer, lr_min=1e-6, lr_max=10.0, beta=0.9):
    # Une seule époque ; le pas est multiplié par q à chaque mini-batch.
    n = len(loader)
    q = (lr_max / lr_min) ** (1 / (n - 1))
    optimizer = make_optimizer(model.parameters(), lr_min)
    lr, avg, best = lr_min, 0.0, float("inf")
    lrs, losses = [], []
    model.train()
    for i, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        for group in optimizer.param_groups:      # on impose le pas de ce mini-batch
            group["lr"] = lr
        optimizer.zero_grad()
        loss = lossfn(model(X), y)
        loss.backward()
        optimizer.step()
        avg = beta * avg + (1 - beta) * loss.item()          # moyenne glissante de la perte
        smoothed = avg / (1 - beta ** (i + 1))                # correction du biais initial (comme Adam)
        lrs.append(lr); losses.append(smoothed)
        best = min(best, smoothed)
        if not math.isfinite(smoothed) or smoothed > 4 * best:   # la perte explose : on s'arrête
            break
        lr = lr * q
    return lrs, losses

rt_loader = make_loaders(Xtr, ytr, Xva, yva, batch_size=16, seed=0)[0]
range_test = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, act in zip(axes, ACTIVATIONS):
    for opt_name, make_opt in OPTIMIZERS.items():
        set_seed(0)
        m = MLP1(activation=act).to(device)
        lrs_rt, losses_rt = lr_range_test(m, rt_loader, nn.CrossEntropyLoss(), make_opt)
        i_min = int(np.argmin(losses_rt))
        range_test[(act, opt_name)] = lrs_rt[i_min]
        ax.semilogx(lrs_rt, losses_rt, label=f"{opt_name} (min : {lrs_rt[i_min]:.1g})")
    ax.set_title(f"LR range test — {act}"); ax.set_xlabel("pas d'apprentissage (échelle log)"); ax.legend(fontsize=8)
axes[0].set_ylabel("perte d'apprentissage lissée")
plt.tight_layout(); save_fig("v1_lr_range_test")

### 5.2 Le balayage du pas d'apprentissage
Pour chaque couple (activation, optimiseur), on entraîne le réseau `N_EPOCHS_SWEEP` époques pour chaque pas d'une
grille logarithmique (cours : *explore scale-type hyperparameters on a logarithmic grid*), avec la même graine.
**Critère de choix** : le F1 macro de validation, moyenné sur les 3 dernières époques (moins sensible aux
oscillations qu'une seule époque).

In [ ]:
LR_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]

def score_fin(h, key="val_f1", k=3):
    v = h[key][-k:]
    return float(np.mean(v)) if all(math.isfinite(x) for x in h["train_loss"]) else 0.0

sweep_rows = []
t0 = time.time()
for act in ACTIVATIONS:
    for opt_name, make_opt in OPTIMIZERS.items():
        for lr in LR_GRID:
            set_seed(0)
            tl, vl = make_loaders(Xtr, ytr, Xva, yva, seed=0)
            m = MLP1(activation=act).to(device)
            h = fit(m, tl, vl, make_opt(m.parameters(), lr), N_EPOCHS_SWEEP, verbose=False)
            sweep_rows.append({"activation": act, "optimiseur": opt_name, "lr": lr, "val F1 (fin)": score_fin(h)})
print(f"balayage : {len(sweep_rows)} entraînements de {N_EPOCHS_SWEEP} époques, {time.time() - t0:.0f} s")
sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(os.path.join(LOG_DIR, "v1_balayage_lr.csv"), index=False)
display(sweep.pivot_table(index=["activation", "optimiseur"], columns="lr", values="val F1 (fin)").round(3))

best_lr = sweep.loc[sweep.groupby(["activation", "optimiseur"])["val F1 (fin)"].idxmax()] \
               .set_index(["activation", "optimiseur"])["lr"].to_dict()
display(pd.DataFrame({"pas retenu (balayage)": best_lr,
                      "minimum du LR range test": range_test}).rename_axis(["activation", "optimiseur"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, act in zip(axes, ACTIVATIONS):
    for opt_name in OPTIMIZERS:
        s = sweep[(sweep.activation == act) & (sweep.optimiseur == opt_name)]
        ax.semilogx(s["lr"], s["val F1 (fin)"], "o-", label=opt_name)
    ax.set_title(act); ax.set_xlabel("pas d'apprentissage"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
axes[0].set_ylabel(f"F1 macro de validation ({N_EPOCHS_SWEEP} époques)")
plt.tight_layout(); save_fig("v1_balayage_lr")

**Question 5.2.1** — Décrivez les trois zones des courbes du *LR range test*. Le balayage confirme-t-il la zone
indiquée par le *LR range test* (rappel du cours : choisir environ un ordre de grandeur sous le minimum) ? Comparez
les pas retenus pour SGD, SGD+momentum et Adam, et reliez l'écart entre SGD et SGD+momentum au « pas effectif »
$\eta/(1-\mu)$ (TP 2, question 1.9.2). L'activation change-t-elle le bon pas ?

*Réponse :*

### 5.3 Activations × optimiseurs, chacun avec son pas, sur 3 graines (parties 2.2 et 2.3)
Chaque configuration est entraînée `N_EPOCHS` époques avec **3 graines** différentes (initialisation et ordre des
mini-batches). On garde les historiques (courbes) et les modèles entraînés.

In [ ]:
runs = {}                      # (activation, optimiseur) -> liste de (historique, modèle), une par graine
t0 = time.time()
for act in ACTIVATIONS:
    for opt_name, make_opt in OPTIMIZERS.items():
        lr = best_lr[(act, opt_name)]
        runs[(act, opt_name)] = []
        for seed in SEEDS:
            set_seed(seed)
            tl, vl = make_loaders(Xtr, ytr, Xva, yva, seed=seed)
            m = MLP1(activation=act).to(device)
            h = fit(m, tl, vl, make_opt(m.parameters(), lr), N_EPOCHS, verbose=False)
            runs[(act, opt_name)].append((h, m))
print(f"{len(runs) * len(SEEDS)} entraînements de {N_EPOCHS} époques, {time.time() - t0:.0f} s")

def ms(values):
    # « moyenne ± écart-type » (écart-type empirique, ddof=1)
    v = np.asarray(values, dtype=float)
    return f"{v.mean():.4f} ± {v.std(ddof=1):.4f}"

rows = []
for (act, opt_name), rs in runs.items():
    hs = [h for h, _ in rs]
    rows.append({"activation": act, "optimiseur": opt_name, "lr": best_lr[(act, opt_name)],
                 "train acc": ms([score_fin(h, "train_acc") for h in hs]),
                 "val acc": ms([score_fin(h, "val_acc") for h in hs]),
                 "val F1 macro": ms([score_fin(h) for h in hs]),
                 "meilleure époque (val loss)": ms([h["best_epoch"] for h in hs]),
                 "temps (s)": ms([h["time"] for h in hs]),
                 "_mean": np.mean([score_fin(h) for h in hs]), "_std": np.std([score_fin(h) for h in hs], ddof=1)})
table = pd.DataFrame(rows).sort_values("_mean", ascending=False).reset_index(drop=True)
table.drop(columns=["_mean", "_std"]).to_csv(os.path.join(LOG_DIR, "v1_activations_optimiseurs.csv"), index=False)
display(table.drop(columns=["_mean", "_std"]))
print(f"référence (régression logistique) : val acc {base_val['accuracy']:.4f} | val F1 macro {base_val['f1_macro']:.4f}")

### 5.4 Les écarts sont-ils plus grands que la variabilité ? (partie 2.3)
Le sujet : « un écart entre deux configurations plus petit que cette variabilité ne permet pas de conclure ».
On applique une règle simple et explicite : la meilleure configuration n'est déclarée **meilleure** qu'une autre que si
$|\bar{F}_1 - \bar{F}_2| > s_1 + s_2$, où $s$ est l'écart-type sur les 3 graines. La même règle sert pour la
comparaison avec la référence (dont la variabilité est nulle : la régression logistique est déterministe).

In [ ]:
best = table.iloc[0]
cmp = []
for _, r in table.iloc[1:].iterrows():
    d, tol = best["_mean"] - r["_mean"], best["_std"] + r["_std"]
    cmp.append({"comparée à": f"{r['activation']} / {r['optimiseur']}", "écart de F1": round(d, 4),
                "s1 + s2": round(tol, 4), "conclusion": "écart significatif" if d > tol else "on ne peut pas conclure"})
d_base = best["_mean"] - base_val["f1_macro"]
cmp.append({"comparée à": "référence (régression logistique)", "écart de F1": round(d_base, 4),
            "s1 + s2": round(best["_std"], 4),
            "conclusion": "écart significatif" if abs(d_base) > best["_std"] else "on ne peut pas conclure"})
print(f"meilleure configuration (validation) : {best['activation']} / {best['optimiseur']} (lr = {best['lr']:g})")
pd.DataFrame(cmp)

**Question 5.4.1** — Quelle activation et quel optimiseur donnent les meilleurs résultats ? Parmi les écarts observés,
lesquels sont réellement concluants ? Le MLP fait-il mieux que la référence, au-delà de la variabilité ?
Commentez la stabilité (écarts-types) selon l'optimiseur.

*Réponse :*

### 5.5 Accuracy ou F1-score ?
L'**accuracy** compte la proportion de bonnes réponses, toutes classes confondues : une classe fréquente pèse plus.
Le **F1 macro** calcule le F1 (moyenne harmonique de la précision et du rappel) de **chaque** classe, puis en fait la
moyenne sans pondération : chaque classe pèse autant. Le **F1 par classe** montre où se trouvent les erreurs.

In [ ]:
best_key = (best["activation"], best["optimiseur"])
h_best, m_best = runs[best_key][0]                               # graine 0
yv_true, yv_pred = predict(m_best, val_loader, device)
display(pd.DataFrame({
    "part des fenêtres (val)": np.bincount(yva, minlength=6) / len(yva),
    "F1 MLP retenu": f1_score(yv_true, yv_pred, average=None),
    "F1 référence": f1_score(yva, yva_pred_base, average=None),
}, index=CLASSES).round(3))

acc_f1 = pd.DataFrame([{"accuracy": score_fin(h, "val_acc"), "F1 macro": score_fin(h)}
                       for rs in runs.values() for h, _ in rs])
print(f"sur les {len(acc_f1)} entraînements : écart maximal |accuracy − F1 macro| = "
      f"{(acc_f1['accuracy'] - acc_f1['F1 macro']).abs().max():.4f} ; corrélation = {acc_f1.corr().iloc[0, 1]:.3f}")

**Question 5.5.1** — Quelle métrique est la plus informative ici, et pourquoi ? Votre réponse dépend-elle de la
distribution des classes observée dans l'exploration (partie 3 de `00_exploration`) ? Que se passerait-il avec une
classe rare ?

*Réponse :*

### 5.6 Retour sur la partie 1.3 avec le MLP
On refait la comparaison des deux découpages avec la configuration retenue (3 graines) : le découpage aléatoire
donne-t-il aussi une estimation plus optimiste pour le MLP ? Les modèles sont gardés pour la partie 7.

In [ ]:
act_b, opt_b = best_key
lr_b = best_lr[best_key]
runs_random_split = []
for seed in SEEDS:
    set_seed(seed)
    tl, vl = make_loaders(Xtr_r, ytr_r, Xva_r, yva_r, seed=seed)
    m = MLP1(activation=act_b).to(device)
    h = fit(m, tl, vl, OPTIMIZERS[opt_b](m.parameters(), lr_b), N_EPOCHS, verbose=False)
    runs_random_split.append((h, m))

split_val[("MLP retenu", "aléatoire")] = np.mean([score_fin(h, "val_acc") for h, _ in runs_random_split])
split_val[("MLP retenu", "par volontaire")] = np.mean([score_fin(h, "val_acc") for h, _ in runs[best_key]])
pd.Series(split_val, name="accuracy de validation (moyenne)").round(4).to_frame()

### 5.7 La configuration retenue
Tous les choix sont faits sur la validation. La configuration retenue est enregistrée : elle sera réutilisée par
`06_interpretability.ipynb` (partie 6 du sujet) et dans le rapport.

In [ ]:
retenue = {"hidden_dim": 64, "activation": act_b, "optimizer": opt_b, "lr": lr_b, "n_epochs": N_EPOCHS,
           "val_subjects": [int(s) for s in val_subjects],
           "val_acc": table.iloc[0]["val acc"], "val_f1": table.iloc[0]["val F1 macro"]}
with open(os.path.join(LOG_DIR, "v1_configuration_retenue.json"), "w") as f:
    json.dump(retenue, f, indent=2)
retenue

---
## Partie 6 — Visualisation et comparaison des résultats

### 6.1 Courbes d'apprentissage de la configuration retenue
Cours (*Reading learning curves*) : on lit l'évolution de la perte et de l'accuracy en apprentissage (tirets) et en
validation (traits pleins), pour les 3 graines.

In [ ]:
plot_curves({f"graine {s}": h for s, (h, _) in zip(SEEDS, runs[best_key])},
            title=f"MLP1 {act_b} / {opt_b} (lr = {lr_b:g})", name="v1_courbes_configuration_retenue")
plot_curves({f"graine {s}": h for s, (h, _) in zip(SEEDS, runs[best_key])},
            key_pairs=(("train_loss", "val_loss"), ("train_f1", "val_f1")), title="F1 macro")

### 6.2 Comparaison des optimiseurs et des activations (graine 0)

In [ ]:
for act in ACTIVATIONS:
    plot_curves({f"{o} (lr = {best_lr[(act, o)]:g})": runs[(act, o)][0][0] for o in OPTIMIZERS},
                title=f"activation {act}", name=f"v1_courbes_{act}")

In [ ]:
plt.figure(figsize=(8, 4))
for i, (key, rs) in enumerate(runs.items()):
    v = np.array([h["val_f1"] for h, _ in rs])                   # (graines, époques)
    ep = np.arange(1, v.shape[1] + 1)
    plt.plot(ep, v.mean(axis=0), c=f"C{i}", label=f"{key[0]} / {key[1]}")
    plt.fill_between(ep, v.mean(axis=0) - v.std(axis=0), v.mean(axis=0) + v.std(axis=0), color=f"C{i}", alpha=0.15)
plt.axhline(base_val["f1_macro"], ls="--", c="k", label="référence")
bas = min(base_val["f1_macro"], min(np.mean([h["val_f1"][-1] for h, _ in rs]) for rs in runs.values())) - 0.05
plt.xlabel("époque"); plt.ylabel("F1 macro de validation"); plt.ylim(max(0, bas), 1.0); plt.legend(fontsize=7, ncol=2)
plt.title("Moyenne ± écart-type sur 3 graines"); save_fig("v1_f1_validation_9_configurations")

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(yv_true, yv_pred, normalize="true"), display_labels=CLASSES).plot(
    cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=45)
plt.title(f"MLP1 {act_b} / {opt_b} — matrice de confusion (validation)"); save_fig("v1_mlp_confusion_val")

**Question 6.2.1** — Commentez vos observations (TP 2, partie 4 : grille de diagnostic) :
- **convergence** : quels optimiseurs convergent le plus vite ? Avec quelle activation la convergence est-elle la plus
  lente, et pourquoi (pensez à la dérivée de la sigmoïde, $\sigma'\le 1/4$, cours : *vanishing gradients*) ?
- **stabilité** : les courbes de validation oscillent-elles ? Les graines donnent-elles des résultats proches ?
- **sous/sur-apprentissage** : l'accuracy d'apprentissage atteint-elle 100 % ? La perte de validation remonte-t-elle
  alors que celle d'apprentissage continue de baisser ? À partir de quelle époque (`meilleure époque`) ?
- quelles classes sont confondues sur la validation ?

*Réponse :*

---
## Partie 7 — Évaluation finale sur le jeu de test

Tous les choix sont maintenant figés. On évalue **une seule fois** sur `test.csv` (des volontaires jamais vus) :
- la **référence** (régression logistique) ;
- le **MLP retenu**, pour ses 3 graines (moyenne ± écart-type) ;
- pour conclure la partie 1.3, les modèles entraînés avec le **découpage aléatoire**.

Les entrées du test sont standardisées avec les statistiques **de l'apprentissage** ($\mu$, $\sigma$ de la section 3.2).
La comparaison avec la version 2 (réseaux profonds) est faite dans le rapport.

In [ ]:
df_test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
X_test_raw = df_test[features].to_numpy(dtype=np.float32)
y_test = df_test["Activity"].map(label_to_int).to_numpy(dtype=np.int64)
X_test = ((X_test_raw - mu_x) / sd_x).astype(np.float32)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test, dtype=torch.long)), batch_size=1024)
print(f"test : {len(y_test)} fenêtres, {df_test['subject'].nunique()} volontaires")

In [ ]:
yte_pred_base = baseline.predict(X_test)
base_test = {"accuracy": accuracy_score(y_test, yte_pred_base), "f1_macro": f1_score(y_test, yte_pred_base, average="macro")}
mlp_test = [evaluate(m, test_loader, lossfn, device) for _, m in runs[best_key]]       # (perte, acc, F1) par graine

resultats = pd.DataFrame({
    "référence (régression logistique)": {
        "val accuracy": f"{base_val['accuracy']:.4f}", "test accuracy": f"{base_test['accuracy']:.4f}",
        "val F1 macro": f"{base_val['f1_macro']:.4f}", "test F1 macro": f"{base_test['f1_macro']:.4f}"},
    f"MLP1 {act_b} / {opt_b} (3 graines)": {
        "val accuracy": table.iloc[0]["val acc"], "test accuracy": ms([r[1] for r in mlp_test]),
        "val F1 macro": table.iloc[0]["val F1 macro"], "test F1 macro": ms([r[2] for r in mlp_test])},
}).T
display(resultats)

In [ ]:
yt_true, yt_pred = predict(runs[best_key][0][1], test_loader, device)
display(pd.DataFrame({"F1 référence (test)": f1_score(y_test, yte_pred_base, average=None),
                      "F1 MLP retenu (test, graine 0)": f1_score(yt_true, yt_pred, average=None)}, index=CLASSES).round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, (titre, yp) in zip(axes, [("Référence", yte_pred_base), (f"MLP1 {act_b} / {opt_b}", yt_pred)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, yp, normalize="true"), display_labels=CLASSES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=45)
    ax.set_title(f"{titre} — matrice de confusion (test)")
plt.tight_layout(); save_fig("v1_confusion_test")

### 7.1 Conclusion de la partie 1.3 : quel découpage de validation prédit le mieux le test ?

In [ ]:
X_test_r = ((X_test_raw - mu_r) / sd_r).astype(np.float32)          # statistiques du découpage aléatoire
test_loader_r = DataLoader(TensorDataset(torch.tensor(X_test_r), torch.tensor(y_test, dtype=torch.long)), batch_size=1024)
test_acc = {
    ("régression logistique", "aléatoire"): accuracy_score(y_test, baseline_r.predict(X_test_r)),
    ("régression logistique", "par volontaire"): base_test["accuracy"],
    ("MLP retenu", "aléatoire"): np.mean([evaluate(m, test_loader_r, lossfn, device)[1] for _, m in runs_random_split]),
    ("MLP retenu", "par volontaire"): np.mean([r[1] for r in mlp_test]),
}
p13 = pd.DataFrame({"accuracy validation": split_val, "accuracy test": test_acc})
p13["validation − test"] = p13["accuracy validation"] - p13["accuracy test"]
display(p13.round(4))

with open(os.path.join(LOG_DIR, "v1_resultats_test.json"), "w") as f:
    json.dump({"resultats": resultats.to_dict(),
               "decoupages": {f"{a} | {b}": {k: float(v) for k, v in row.items()} for (a, b), row in p13.iterrows()}},
              f, indent=2)

**Question 7.1.1** — Les scores de test sont-ils proches des scores de validation (par volontaire) ? Que deviendrait
la performance annoncée si l'on essayait maintenant d'autres configurations en regardant le test (TP 2,
question 3.3.2) ?

**Question 7.1.2** — Partie 1.3 : quel découpage de validation donne l'estimation la plus proche du test ? Concluez sur
la méthodologie.

**Question 7.1.3** — Quelles classes sont les plus confondues sur le test ? La confusion est-elle plausible du point
de vue des capteurs (téléphone porté à la ceinture : que mesure-t-il quand on est assis ou debout, immobile) ?
Le MLP améliore-t-il la référence sur ces classes ?

*Réponse :*